# 01 – Explorative Datenanalyse (EDA)

**Porto Seguro’s Safe Driver Prediction – Aicha**

Die EDA wurde eigenständig durchgeführt. Ziel ist es, aus der Datenstruktur konkrete Entscheidungen für die spätere Modellierung abzuleiten.

In [ ]:
import sys
sys.path.append('../../src')
from data_utils import load_arff, describe_features
import pandas as pd
import matplotlib.pyplot as plt

df = load_arff('../../data/dataset.arff')
print('Shape:', df.shape)
df.head()

In [ ]:
bin_cols, cat_cols, ordinal_cols, cont_cols, calc_cols = describe_features(df)
print('Binär:', len(bin_cols))
print('Kategorisch:', len(cat_cols))
print('Ordinal:', len(ordinal_cols))
print('Kontinuierlich:', len(cont_cols))
print('Calc:', len(calc_cols))

## Klassenverteilung

Die Zielvariable ist stark unausgeglichen. In der bisherigen Analyse ergeben sich ungefähr **96,4 % Klasse 0** und **3,6 % Klasse 1**. Daher wird Accuracy nicht als alleinige Bewertungsmetrik verwendet.

In [ ]:
counts = df['target'].value_counts().sort_index()
print(counts)
print((counts / len(df) * 100).round(2))
counts.plot(kind='bar', figsize=(5,4))
plt.xlabel('Zielvariable')
plt.ylabel('Anzahl Beobachtungen')
plt.title('Klassenverteilung')
plt.tight_layout()
plt.show()

## Fehlende Werte

Die stärksten Fehlquoten liegen bei `ps_car_03_cat` (ca. 69,1 %) und `ps_car_05_cat` (ca. 44,8 %). Diese beiden Features werden in der Baseline-Modellierung entfernt.

In [ ]:
missing = ((df.drop(columns=['target']) == -1).mean() * 100).sort_values(ascending=False)
display(missing.head(15).to_frame('missing_percent'))
missing.head(15).sort_values().plot(kind='barh', figsize=(9,6))
plt.xlabel('Anteil fehlender Werte (%)')
plt.title('Fehlende Werte – Top 15 Features')
plt.tight_layout()
plt.show()

## Korrelationen

Die linearen Korrelationen mit `target` sind insgesamt schwach. Die höchste beobachtete Korrelation liegt bei ungefähr **0,054 für `ps_car_13`**. Dies unterstützt die Untersuchung nichtlinearer Tree-Based Models.

In [ ]:
corr = df[cont_cols + ordinal_cols + ['target']].corr()['target'].drop('target')
corr = corr.sort_values(key=abs, ascending=False)
display(corr.head(15).to_frame('correlation'))
corr.head(15).sort_values().plot(kind='barh', figsize=(9,6))
plt.xlabel('Korrelation mit Zielvariable')
plt.title('Top-Korrelationen mit target')
plt.tight_layout()
plt.show()

## EDA → Modellierungsentscheidung

1. Starkes Klassenungleichgewicht → `scale_pos_weight` und zusätzlich SMOTE testen.
2. Sehr hohe Fehlquoten bei zwei Features → diese in der Baseline entfernen.
3. Schwache lineare Korrelationen → Tree-Based Models als Schwerpunkt untersuchen.
4. Fehlende Werte konsequent und reproduzierbar behandeln.